# CUDA SfmLevenbergMarquardtOptimizer

> **Created by Codex.**

Run BAL-style bundle adjustment with device-resident linearization, Schur elimination, solving, retraction, and error evaluation.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

## Elimination and backend selection

CUDA exposes the same `SfmEliminationMode` axis as the CPU optimizer. Schur is the default and uses the existing bespoke CUDA landmark-elimination kernels. Full is reserved for a later implementation and currently throws immediately at the optimizer boundary; it never falls back to a CPU-linearized solve.

The reduced camera system can be solved with dense Cholesky, cuDSS, or PCG. Dense Cholesky remains the default.

In [ ]:
import gtsam

if hasattr(gtsam, "cuda"):
    params = gtsam.cuda.SfmLevenbergMarquardtParams.ceresDefaults()
    params.setEliminationMode(gtsam.SfmEliminationMode.Schur)
    params.setLinearSolver(gtsam.cuda.LinearSolverType.Cudss)
    print(params.getEliminationMode(), params.getLinearSolver())

## Ordering and supported graphs

A CUDA Schur ordering contains camera keys only. Backends that do not consume an ordering reject one. The graph must contain supported BAL projection or point-batched projection factors over `SfmCamera` and `Point3` values. Other values are carried through unchanged.

In [ ]:
# optimizer = gtsam.cuda.SfmLevenbergMarquardtOptimizer(
#     graph, initial, params
# )
# result = optimizer.optimize()
# diagnostics = optimizer.result()

## Reserved Full hook

Setting `SfmEliminationMode.Full` is useful for feature detection and configuration round trips, but optimization raises: `CUDA SFM Full elimination mode is not implemented; select Schur.` The public API will not need to change when full-system CUDA solving is added.